# Sectoral IAM for modelling global SAI

## Model

In [ ]:
# Libraries
import scipy.optimize as optimize
import numpy as np
import pandas as pd

In [ ]:
# Model functions

def model_to_i(model):
  '''
  model_to_i(str model)

  Transforms model to number

  Inputs:
  - model (str): GCM model name ["CESM", "IPSL", "MPI", "UKESM"]
  
  Outputs:
  - int: model index in lists
  '''

  if   model=="CESM":  return 0
  elif model=="IPSL":  return 1
  elif model=="MPI":   return 2
  elif model=="UKESM": return 3



def calc_GWP(emissions, model="CESM"):
  '''
  calc_GWP(float emisiones, str model="CESM")

  Calculates SO2 greenhouse warming potential

  Inputs:
  - emissions (float): SO2 emissions [Tg SO2/yr]
  - model (str):       GCM model name ["CESM", "IPSL", "MPI", "UKESM"]
  
  Outputs:
  - SO2_GWP:           SO2 greenhouse warming potential
  '''

  # Coefficients for each climate model
  b0=[-3504.9, -3784.2, -2054.2, -2782.5]
  b1=[549.27, 739.53, 241.55, 244.86]
  i=model_to_i(model)
  # Logarithmic regression
  SO2_GWP=b0[i] + b1[i]*np.log(emissions)
  return SO2_GWP

def calc_RF(emissions, RF_obj=-1, model="CESM"):
  '''
  calc_RF(float emissions, float RF_obj=-1, str model="CESM"):

  Calculates SO2 radiative forcing given the emissions. The function substracts
  an objective radiative forcing for numerical solution.

  Inputs:
  - emissions (float): SO2 emissions [Tg SO2/yr]
  - RF_obj (float):    radiative forcing to look for [W/m2]
  - model (str):       GCM model name ["CESM", "IPSL", "MPI", "UKESM"]
  
  Outputs:
  - float:             difference between objective radiative forcing and
                       the forcing calculated using emissions
  '''

  # Coefficients for each climate model
  b0=[-0.2249, -0.2463, -0.1299, -0.1740]
  b1=[0.0332, 0.0447, 0.0146, 0.0148]
  i=model_to_i(model)
  # Logarithmic regression
  FE=b0[i] + b1[i]*np.log(emissions)
  RF=emissions*FE
  # Substract objective for numerical solution
  return RF-RF_obj



def get_RF(x, RF_obj=-1, model="CESM", log_coef=-0.036, max_sai=100,
  fixed_GWP=None, sulfur_cost=0.22*5, fuel_cost=0.77155*0.15):
  '''
  get_RF(tuple of floats x, float RF_obj=-1, str model="CESM",
    str log_coef=-0.036, str max_sai=100, float fixed_GWP=None,
    float sulfur_cost=0.22*5, float fuel_cost=0.77155*0.15)

  Calculates SO2 radiative forcing given the emissions and CO2e price. The
  function substracts an objective radiative forcing for numerical solution.

  Inputs:
  - tuple x:           a tuple of:
                       - float: SO2 emissions [Tg SO2/yr]
                       - float: CO2e price [1975 USD/t CO2e]
  - RF_obj (float):    radiative forcing to look for [W/m2]
  - model (str):       GCM model name ["CESM", "IPSL", "MPI", "UKESM"]
  - float log_coef:    Logit discrete choice function coefficient
  - float max_sai:     maximum SAI emissions [Tg SO2/yr]
  - float fixed_GWP:   None if variable GWP, float for a fixed emissions to
                       calculate GWP
  - float sulfur_cost: sulfur aerosol cost [1975 USD/kg SO2]
  - float fuel_cost:   SAI fuel costs [1975 USD/t delivered SO2]
  
  Outputs:
  - float:             difference between objective radiative forcing and
                       the forcing calculated using emissions
  '''

  cprice = x[0]
  rf = x[1]
  # Calculate emissions numerically
  emissions=optimize.root_scalar(
    calc_RF, args=(rf, model), method="newton", x0=1).root
  # Fixed or variable GWP
  if fixed_GWP is None: SO2_GWP=calc_GWP(emissions, model)
  else:                 SO2_GWP=calc_GWP(1,         model)
  # Cost integration
  SO2_price=cprice*SO2_GWP/2.212
  aerosol_cost=sulfur_cost+0.03
  other_co=0.77155*0.85-(0.22+0.03)
  total_cost=(other_co+aerosol_cost+fuel_cost)*1000+SO2_price
  # Logit choice function
  cost_no_sai=0.001
  SAI_share=(np.exp(log_coef*total_cost)
    /(np.exp(log_coef*total_cost)+np.exp(log_coef*cost_no_sai)))
  max_sai=100
  total_sai=SAI_share*max_sai
  # Fixed or variable GWP
  if fixed_GWP is None: FE=0.0332*np.log(total_sai)-0.2249 
  else:                 FE=0.0332*np.log(1)-0.2249 
  RF=total_sai*FE
  # Substract objective for numerical solution
  return np.abs(RF-RF_obj)



def solve_rf(RF_obj=-1, model="CESM", log_coef=-0.036, max_sai=100,
  fixed_GWP=None, sulfur_cost=0.22*5, fuel_cost=0.77155*0.15, tol=0.001):
  '''
  solve_RF(float RF_obj=-1, str model="CESM",
    str log_coef=-0.036, str max_sai=100, float fixed_GWP=None,
    float sulfur_cost=0.22*5, float fuel_cost=0.77155*0.15, float tol=0.001)

  Calculate the CO2e price to achieve a desired radiative forcing with SAI.

  Inputs:
  - RF_obj (float):    radiative forcing to look for [W/m2]
  - model (str):       GCM model name ["CESM", "IPSL", "MPI", "UKESM"]
  - float log_coef:    Logit discrete choice function coefficient
  - float max_sai:     maximum SAI emissions [Tg SO2/yr]
  - float fixed_GWP:   None if variable GWP, float for a fixed emissions to
                       calculate GWP
  - float sulfur_cost: sulfur aerosol cost [1975 USD/kg SO2]
  - float fuel_cost:   SAI fuel costs [1975 USD/t delivered SO2]
  - float tol:         solution tolerance
  
  Outputs:
  - float solution:    CO2e price [1075 USD/t CO2e]
  '''

  # Predefine function with its args and kwargs
  fun=lambda x: get_RF(x, RF_obj=RF_obj,
    model="CESM", log_coef=-0.036, max_sai=max_sai,
    fixed_GWP=fixed_GWP, sulfur_cost=sulfur_cost, fuel_cost=fuel_cost)
  # We brute force over solution boundaries since the derivative is not well
  # behaved over the vincinity of solutions.
  step=tol/100
  x_0=0-step
  fun_res=tol*2
  while (fun_res>tol) and (x_0<15):
    x_0=x_0+step
    x_1=x_0+step
    # Solve the problem as a constrained minimization which should result in 0,
    # but on a local region of the function it might not be able to converge
    # to the solution
    solution = optimize.minimize(fun, [0,RF_obj], bounds=[(x_0,x_1),
      (RF_obj-tol, RF_obj+tol)], tol=tol)
    fun_res=solution.fun
    # Print brute force iteration results
    #print(f"{x_0:.3f}",f"{x_1:.3f}", f"{fun_res:.6f}")
  return solution



def get_cprice(x, cprice=1, model="CESM", log_coef=-0.036, max_sai=100,
  fixed_GWP=None, sulfur_cost=0.22*5, fuel_cost=0.77155*0.15):
  '''
  get_cprice(tuple of float x, float RF_obj=-1, str model="CESM",
    str log_coef=-0.036, str max_sai=100, float fixed_GWP=None,
    float sulfur_cost=0.22*5, float fuel_cost=0.77155*0.15)

  A simpler version of get_RF, planned when one knows the carbon price and 
  wishes to understand the emissions achieved with that price.

  Inputs:
  - tuple x:           a tuple of:
                       - float: SO2 emissions [Tg SO2/yr]
  - RF_obj (float):    radiative forcing to look for [W/m2]
  - model (str):       GCM model name ["CESM", "IPSL", "MPI", "UKESM"]
  - float log_coef:    Logit discrete choice function coefficient
  - float max_sai:     maximum SAI emissions [Tg SO2/yr]
  - float fixed_GWP:   None if variable GWP, float for a fixed emissions to
                       calculate GWP
  - float sulfur_cost: sulfur aerosol cost [1975 USD/kg SO2]
  - float fuel_cost:   SAI fuel costs [1975 USD/t delivered SO2]
  
  Outputs:
  - float:             difference between objective radiative forcing and
                       the forcing calculated using emissions
  '''

  emissions = x[0]
  # Fixed or variable GWP
  if fixed_GWP is None: SO2_GWP=calc_GWP(emissions, model)
  else:                 SO2_GWP=calc_GWP(1,         model)
  # Cost integration
  SO2_price=cprice*SO2_GWP/2.212
  aerosol_cost=sulfur_cost+0.03
  other_co=0.77155*0.85-(0.22+0.03)
  total_cost=(other_co+aerosol_cost+fuel_cost)*1000+SO2_price
  # Logit choice function
  cost_no_sai=0.001
  SAI_share=(np.exp(log_coef*total_cost)
    /(np.exp(log_coef*total_cost)+np.exp(log_coef*cost_no_sai)))
  total_sai=SAI_share*max_sai
  # Substract objective for numerical solution
  return np.abs(total_sai-emissions)



def solve_cprice(cprice=1, model="CESM", log_coef=-0.036, max_sai=100,
  fixed_GWP=None, sulfur_cost=0.22*5, fuel_cost=0.77155*0.15, tol=0.001):
  '''
  solve_cprice(float cprice=1, str model="CESM",
    str log_coef=-0.036, str max_sai=100, float fixed_GWP=None,
    float sulfur_cost=0.22*5, float fuel_cost=0.77155*0.15, float tol=0.001)

  Calculate the emissions that result from a given CO2 price.

  Inputs:
  - cprice (float):    CO2e price [1975 USD/t CO2e]
  - model (str):       GCM model name ["CESM", "IPSL", "MPI", "UKESM"]
  - float log_coef:    Logit discrete choice function coefficient
  - float max_sai:     maximum SAI emissions [Tg SO2/yr]
  - float fixed_GWP:   None if variable GWP, float for a fixed emissions to
                       calculate GWP
  - float sulfur_cost: sulfur aerosol cost [1975 USD/kg SO2]
  - float fuel_cost:   SAI fuel costs [1975 USD/t delivered SO2]
  - float tol:         solution tolerance
  
  Outputs:
  - float solution:    SO2 emissions [Tg SO2/yr]
  '''

  # Predefine function with its args and kwargs
  fun=lambda x: get_cprice(x, cprice=cprice,
    model="CESM", log_coef=log_coef, max_sai=max_sai,
    fixed_GWP=fixed_GWP, sulfur_cost=sulfur_cost, fuel_cost=fuel_cost)
  # Solve the problem as a constrained minimization which should result in 0
  emiss = optimize.minimize(fun, [10], bounds=[(0.00001,100)], tol=tol).x[0]

  return emiss

def setup_model(model="CESM", initial_depletion=12500, sulfur_elasticity=0.5,
  fuel_change=-0.0082, fossil_change=0.0085, fossil_fraction=0.9,
  demand_base=80.0, demand_change=0.018, years=np.arange(2020, 2110, 10),
  forcings=[-0.077795901, -0.202776003, -0.377084028, -0.625278416,
  -0.945167262, -1.320845384, -1.737155925, -2.155592258, -2.5283355]):
  '''
  setup_model(str model="CESM", float initial_depletion=12500,
    float sulfur_elasticity=0.5, float fuel_change=-0.0082,
    float fossil_change=0.0085, float fossil_fraction=0.9,
    float demand_base=80.0, float demand_change=0.018, 
    iterable of ints years=np.arange(2020, 2110, 10),
    iteratble of floats forcings=[-0.077795901, -0.202776003,
    -0.377084028, -0.625278416, -0.945167262, -1.320845384,
    -1.737155925, -2.155592258, -2.5283355])

  Creates a Pandas DataFrame and populates it with the exogenous and endogenous
  demand and prices data, preparing it to be run to find carbon prices.

  Inputs:
  - model (str):                    GCM model name ["CESM", "IPSL", "MPI",
                                    "UKESM"]
  - initial_depletion (float):      historical cumulative extraction of mined
                                    sulfur
  - sulfur_elasticity (float):      price elasticity of sulfur price
  - fuel_change (float):            fractional increase in fuel price per year
  - fossil_change (float):          fractional increase in fossil production
                                    per year
  - fossil_fraction (float):        fraction of sulfur demand met through
                                    fossil reformation
  - demand_base (float):            sulfur demand in initial year
  - demand_change (float):          fractional increase in sulfur demand per
                                    year
  - years (iterable of ints):       years of each period
  - forcings (iteratble of floats): forcing to achieve in each period, must
                                    match the length of years [W/m2]

  Output:
  - df (Pandas DataFrame):            DataFrame populated with scenario data
  '''
  
  # Base sulfur supply
  fossil_base=demand_base*fossil_fraction
  mining_base=demand_base*(1-fossil_fraction)
  
  # Deflator
  US_2020_1975=0.2583022317

  # Create the Pandas DataFrame and populated with exogenous forcing data
  df = pd.DataFrame(columns=["Year", "Radiative forcing [W/m2]",
    "Emissions [Tg SO2/yr]", "CO2 equivalent [GWP]",
    "Cumulative depletion [Tg SO2] (no demand feedback)"])
  year_0=years[0]
  df["Year"] = years
  years_delta = df["Year"] - df["Year"].shift(1)
  years_delta.loc[0] = df.loc[0, "Year"] - year_0
  df = df.set_index("Year")
  df["Radiative forcing [W/m2]"] = forcings

  # Calculate emissions that match the radiative forcing
  df["Emissions [Tg SO2/yr]"] = [optimize.root_scalar(calc_RF, args=(x),
    method="newton", x0=1).root for x in df["Radiative forcing [W/m2]"]]
  df["CO2 equivalent [GWP]"] = calc_GWP(df["Emissions [Tg SO2/yr]"], model)
  
  # Project sulfur supply, demand
  df["Fossil supply [Mt S/yr]"] = (
    fossil_base * (1 + fossil_change)**(df.index-df.index[0]))
  df["Sulfur demand [Mt S/yr]"] = (
    demand_base * (1 + demand_change)**(df.index-df.index[0]))
  df["Mining supply [Mt S/yr]"] = (df["Emissions [Tg SO2/yr]"]/2
    + df["Sulfur demand [Mt S/yr]"] - df["Fossil supply [Mt S/yr]"])
  df["Cumulative depletion [Tg SO2] (no demand feedback)"] = (
    years_delta.values*df["Emissions [Tg SO2/yr]"].shift(1)/2).cumsum()

  # Project sulfur depletion with several cases
  df.loc[years[0], "Cumulative depletion [Tg SO2] (no demand feedback)"] = 0
  df["Cumulative depletion [Tg SO2] (no demand feedback)"] += (
    df["Emissions [Tg SO2/yr]"]/2+initial_depletion+
    mining_base*(df.index-df.index[0]+1))
  df["Cumulative depletion [Tg SO2] (demand feedback)"] = (
    df["Emissions [Tg SO2/yr]"]+initial_depletion+
    df["Mining supply [Mt S/yr]"]*(df.index-df.index[0]+1))
  
  # Project sulfur marginal costs with several cases
  df["Marginal sulfur cost (constant) [2020 USD/kg S]"] = (
    0.22*5/US_2020_1975)
  df["Marginal sulfur cost (no demand feedback) [2020 USD/kg S]"] = (
    df["Marginal sulfur cost (constant) [2020 USD/kg S]"]*
    (df["Cumulative depletion [Tg SO2] (no demand feedback)"]
    /initial_depletion)**(1/sulfur_elasticity))
  df["Marginal sulfur cost (demand feedback) [2020 USD/kg S]"] = (
    df["Marginal sulfur cost (constant) [2020 USD/kg S]"]*
    (df["Cumulative depletion [Tg SO2] (demand feedback)"]
    /initial_depletion)**(1/sulfur_elasticity))

  # Project sulfur costs
  df["Fuel cost [2020 USD/kg delivered SO2]"] = (
    0.77155*0.15/US_2020_1975)
  df["Fuel cost (changing) [2020 USD/kg delivered SO2]"] = (
    0.77155*0.15/US_2020_1975*(1+fuel_change)**(df.index-df.index[0]+1))
  
  return df



def calc_scenario(df, cases=[True, True, True, True, True]):
  '''
  calc_scenario(Pandas DataFrame df,
    list of bools cases=[True, True, True, True, True])

  Runs scenario and enables one to choose among the following cases with
  accumulating complexities:
  0. GWP fixed to 1 Tg
  1. variable GWP
  2. SAI demand feedback
  3. sulfur demand feedback
  4. variable fuel cost

  Inputs:
  - df (Pandas DataFrame): DataFrame populated with scenario data
  - cases (list of bools): flags to determine which cases to run
  
  Output:
  - df (Pandas DataFrame): DataFrame with scenario results
  '''
  
  # Deflator
  US_2020_1975=0.2583022317

  # Solve every period
  for row in df.iterrows():
    # Get costs and radiative forcing from table and deflate
    rf = row[1]["Radiative forcing [W/m2]"]
    sulfur_cost_const = (row[1][
      "Marginal sulfur cost (constant) [2020 USD/kg S]"]
      *US_2020_1975)
    sulfur_cost_sai = (row[1][
      "Marginal sulfur cost (no demand feedback) [2020 USD/kg S]"]
      *US_2020_1975)
    sulfur_cost_demand = (row[1][
      "Marginal sulfur cost (demand feedback) [2020 USD/kg S]"]
      *US_2020_1975)
    fuel_cost_constant=(row[1][
      "Fuel cost (constant) [2020 USD/kg delivered SO2]"]
      *US_2020_1975)
    fuel_cost_variable=(row[1][
      "Fuel cost (increasing) [2020 USD/kg delivered SO2]"]
      *US_2020_1975)
    
    # Solve cases as requested
    if cases[0]:
      df["Carbon price [$2020/tCO2e] (GWP fixed to 1 Tg)"] = (
        solve_rf(rf, fixed_GWP=1, sulfur_cost=sulfur_cost_const,
        fuel_cost=fuel_cost_constant).x[0]/US_2020_1975)
    if cases[1]:
      df["Carbon price [$2020/tCO2e] (variable GWP)"] = (
        solve_rf(rf, sulfur_cost=sulfur_cost_const,
        fuel_cost=fuel_cost_constant).x[0]/US_2020_1975)
    if cases[2]:
      df["Carbon price [$2020/tCO2e] (SAI demand feedback)"] = (
        solve_rf(rf, sulfur_cost=sulfur_cost_sai,
        fuel_cost=fuel_cost_constant).x[0]/US_2020_1975)
    if cases[3]:
      df["Carbon price [$2020/tCO2e] (sulfur demand feedback)"] = (
        solve_rf(rf, sulfur_cost=sulfur_cost_demand,
        fuel_cost=fuel_cost_constant).x[0]/US_2020_1975)
    if cases[4]:
      df["Carbon price [$2020/tCO2e] (variable fuel cost)"] = (
        solve_rf(rf, sulfur_cost=sulfur_cost_demand,
        fuel_cost=fuel_cost_variable).x[0]/US_2020_1975)
    
  return df

## Scenarios (demo)

In [ ]:
# SSP5-7.0

# Initial parameters
model="CESM"
initial_depletion=12500
sulfur_elasticity=0.5
fuel_change=-0.0082
fossil_change=0.0085
fossil_fraction=0.9
demand_base=80
demand_change=0.018
fossil_base=demand_base*fossil_fraction
mining_base=demand_base*(1-fossil_fraction)
years=np.arange(2020, 2110, 10)
forcings=[-0.077795901, -0.202776003, -0.377084028, -0.625278416,
  -0.945167262, -1.320845384, -1.737155925, -2.155592258, -2.5283355]

# Deflactor
US_2020_1975=0.2583022317

df = pd.DataFrame(columns=["Year",
  "Radiative forcing [W/m2]", "Emissions [Tg SO2/yr]", "CO2 equivalent [GWP]",
  "Cumulative depletion [Tg SO2] (no demand feedback)"])
year_0=2020
df["Year"] = years
years_delta = df["Year"] - df["Year"].shift(1)
years_delta.loc[0] = df.loc[0, "Year"] - year_0
df = df.set_index("Year")
df["Radiative forcing [W/m2]"] = forcings
df["Emissions [Tg SO2/yr]"] = [optimize.root_scalar(calc_RF, args=(x),
  method="newton", x0=1).root for x in df["Radiative forcing [W/m2]"]]
df["Fossil supply [Mt S/yr]"] = (
  fossil_base * (1 + fossil_change)**(df.index-df.index[0]))
df["Sulfur demand [Mt S/yr]"] = (
  demand_base * (1 + demand_change)**(df.index-df.index[0]))
df["Mining supply [Mt S/yr]"] = (df["Emissions [Tg SO2/yr]"]/2
  + df["Sulfur demand [Mt S/yr]"] - df["Fossil supply [Mt S/yr]"])
df["Cumulative depletion [Tg SO2] (no demand feedback)"] = (
  years_delta.values*df["Emissions [Tg SO2/yr]"].shift(1)/2).cumsum()

df.loc[years[0], "Cumulative depletion [Tg SO2] (no demand feedback)"] = 0
df["Cumulative depletion [Tg SO2] (no demand feedback)"] += (
  df["Emissions [Tg SO2/yr]"]/2+initial_depletion+
  mining_base*(df.index-df.index[0]+1))

df["Cumulative depletion [Tg SO2] (demand feedback)"] = (
  df["Emissions [Tg SO2/yr]"]+initial_depletion+
  df["Mining supply [Mt S/yr]"]*(df.index-df.index[0]+1))
df["CO2 equivalent [GWP]"] = calc_GWP(df["Emissions [Tg SO2/yr]"], model)
df["Marginal sulfur cost (constant) [2020 USD/kg S]"] = (
  0.22*5/US_2020_1975)
df["Marginal sulfur cost (no demand feedback) [2020 USD/kg S]"] = (
  df["Marginal sulfur cost (constant) [2020 USD/kg S]"]*
  (df["Cumulative depletion [Tg SO2] (no demand feedback)"]/initial_depletion)
  **(1/sulfur_elasticity))
df["Marginal sulfur cost (demand feedback) [2020 USD/kg S]"] = (
  df["Marginal sulfur cost (constant) [2020 USD/kg S]"]*
  (df["Cumulative depletion [Tg SO2] (demand feedback)"]/initial_depletion)
  **(1/sulfur_elasticity))

df["Fuel cost [2020 USD/kg delivered SO2]"] = (
  0.77155*0.15/US_2020_1975)
df["Fuel cost (changing) [2020 USD/kg delivered SO2]"] = (
  0.77155*0.15/US_2020_1975*(1+fuel_change)**(df.index-df.index[0]+1))

df["Carbon price [$2020/tCO2e] (GWP fixed to 1 Tg)"] = [
  solve_rf(x, fixed_GWP=1).x[0]/US_2020_1975
  for x in df["Radiative forcing [W/m2]"]]
'''
df["Carbon price [$2020/tCO2e] (no sulfur demand feedback)"] = [
  solve_rf(x).x[0]/US_2020_1975
  for x in df["Radiative forcing [W/m2]"]]
for row in df.iterrows():
  depletion = row[1]["Cumulative Emissions [Tg SO2]"]
  rf= row[1]["Radiative forcing [W/m2]"]
  df.loc[row[0],
    "Carbon price [$2020/tCO2e] (with SAI sulfur demand feedback)"] = (
    solve_rf(rf, elasticity_aerosol=(sulfur_elasticity, depletion)).x[0]
    /US_2020_1975)
'''

print(f"Results for model: {model}")
df.style.format(thousands=",", precision=2)

Results for model: CESM


,Radiative forcing [W/m2],Emissions [Tg SO2/yr],CO2 equivalent [GWP],Cumulative depletion [Tg SO2] (no demand feedback),Fossil supply [Mt S/yr],Sulfur demand [Mt S/yr],Mining supply [Mt S/yr],Cumulative depletion [Tg SO2] (demand feedback),Marginal sulfur cost (constant) [2020 USD/kg S],Marginal sulfur cost (no demand feedback) [2020 USD/kg S],Marginal sulfur cost (demand feedback) [2020 USD/kg S],Fuel cost (constant) [2020 USD/kg delivered SO2],Carbon price [$2020/tCO2e] (GWP fixed to 1 Tg)
Year,,,,,,,,,,,,,
2020,-0.08,0.29,"-4,179.51","12,508.15",72.00,80.00,8.15,"12,508.44",4.26,4.26,4.26,0.45,3.65
2030,-0.20,0.89,"-3,571.53","12,589.91",78.36,95.62,17.71,"12,695.67",4.26,4.32,4.39,0.45,3.72
2040,-0.38,1.84,"-3,169.07","12,674.81",85.28,114.30,29.94,"13,130.60",4.26,4.38,4.70,0.45,3.76
2050,-0.63,3.39,"-2,834.05","12,764.80",92.81,136.62,45.51,"13,914.06",4.26,4.44,5.28,0.45,3.79
2060,-0.95,5.64,"-2,554.26","12,862.89",101.01,163.31,65.12,"15,175.44",4.26,4.51,6.28,0.45,3.82
2070,-1.32,8.61,"-2,322.43","12,972.60",109.93,195.20,89.57,"17,076.74",4.26,4.59,7.95,0.45,3.85
2080,-1.74,12.26,"-2,128.22","13,097.47",119.64,233.32,119.81,"19,820.66",4.26,4.68,10.71,0.45,3.87
2090,-2.16,16.30,"-1,971.73","13,240.79",130.21,278.89,156.83,"23,651.34",4.26,4.78,15.25,0.45,3.88
2100,-2.53,20.21,"-1,853.65","13,404.25",141.71,333.36,201.75,"28,862.25",4.26,4.90,22.70,0.45,3.90


In [ ]:
# SSP5-4.5 by mitigation and 2.5 additional by SAI (~1.9)

model="CESM"
initial_depletion=12500
sulfur_elasticity=0.5
fuel_change=0.0015
fossil_change=0.0036
fossil_fraction=0.9
demand_base=80
demand_change=0.023
fossil_base=demand_base*fossil_fraction
mining_base=demand_base*(1-fossil_fraction)
years=np.arange(2020, 2110, 10)
forcings=[-0.077795901, -0.202776003, -0.377084028, -0.625278416,
  -0.945167262, -1.320845384, -1.737155925, -2.155592258, -2.5283355]

US_2020_1975=0.2583022317

df = pd.DataFrame(columns=["Year",
  "Radiative forcing [W/m2]", "Emissions [Tg SO2/yr]", "CO2 equivalent [GWP]",
  "Cumulative depletion [Tg SO2] (no demand feedback)"])
year_0=2020
df["Year"] = years
years_delta = df["Year"] - df["Year"].shift(1)
years_delta.loc[0] = df.loc[0, "Year"] - year_0
df = df.set_index("Year")
df["Radiative forcing [W/m2]"] = forcings
df["Emissions [Tg SO2/yr]"] = [optimize.root_scalar(calc_RF, args=(x),
  method="newton", x0=1).root for x in df["Radiative forcing [W/m2]"]]
df["Fossil supply [Mt S/yr]"] = (
  fossil_base * (1 + fossil_change)**(df.index-df.index[0]))
df["Sulfur demand [Mt S/yr]"] = (
  demand_base * (1 + demand_change)**(df.index-df.index[0]))
df["Mining supply [Mt S/yr]"] = (df["Emissions [Tg SO2/yr]"]/2
  + df["Sulfur demand [Mt S/yr]"] - df["Fossil supply [Mt S/yr]"])
df["Cumulative depletion [Tg SO2] (no demand feedback)"] = (
  years_delta.values*df["Emissions [Tg SO2/yr]"].shift(1)/2).cumsum()

df.loc[years[0], "Cumulative depletion [Tg SO2] (no demand feedback)"] = 0
df["Cumulative depletion [Tg SO2] (no demand feedback)"] += (
  df["Emissions [Tg SO2/yr]"]/2+initial_depletion+
  mining_base*(df.index-df.index[0]+1))
df["Cumulative depletion [Tg SO2] (demand feedback)"] = (
  df["Emissions [Tg SO2/yr]"]+initial_depletion+
  df["Mining supply [Mt S/yr]"]*(df.index-df.index[0]+1))
df["CO2 equivalent [GWP]"] = calc_GWP(df["Emissions [Tg SO2/yr]"], model)
df["Marginal sulfur cost (constant) [2020 USD/kg S]"] = (
  0.22*5/US_2020_1975)
df["Marginal sulfur cost (no demand feedback) [2020 USD/kg S]"] = (
  df["Marginal sulfur cost (constant) [2020 USD/kg S]"]*
  (df["Cumulative depletion [Tg SO2] (no demand feedback)"]/initial_depletion)
  **(1/sulfur_elasticity))
df["Marginal sulfur cost (demand feedback) [2020 USD/kg S]"] = (
  df["Marginal sulfur cost (constant) [2020 USD/kg S]"]*
  (df["Cumulative depletion [Tg SO2] (demand feedback)"]/initial_depletion)
  **(1/sulfur_elasticity))

df["Fuel cost (constant) [2020 USD/kg delivered SO2]"] = (
  0.77155*0.15/US_2020_1975)
df["Fuel cost (increasing) [2020 USD/kg delivered SO2]"] = (
  0.77155*0.15/US_2020_1975*(1+fuel_change)**(df.index-df.index[0]+1))

df = df.iloc[[-1]]

for row in df.iterrows():
  rf = row[1]["Radiative forcing [W/m2]"]
  sulfur_cost_const = (row[1][
    "Marginal sulfur cost (constant) [2020 USD/kg S]"]*US_2020_1975)
  sulfur_cost_sai = (row[1][
    "Marginal sulfur cost (no demand feedback) [2020 USD/kg S]"]*US_2020_1975)
  sulfur_cost_demand = (row[1][
    "Marginal sulfur cost (demand feedback) [2020 USD/kg S]"]*US_2020_1975)
  fuel_cost_constant=(row[1][
    "Fuel cost (constant) [2020 USD/kg delivered SO2]"]*US_2020_1975)
  fuel_cost_variable=(row[1][
    "Fuel cost (increasing) [2020 USD/kg delivered SO2]"]*US_2020_1975)
  
  df["Carbon price [$2020/tCO2e] (GWP fixed to 1 Tg)"] = (
    solve_rf(rf, fixed_GWP=1, sulfur_cost=sulfur_cost_const,
    fuel_cost=fuel_cost_constant).x[0]/US_2020_1975)
  df["Carbon price [$2020/tCO2e] (variable GWP)"] = (
    solve_rf(rf, sulfur_cost=sulfur_cost_const,
    fuel_cost=fuel_cost_constant).x[0]/US_2020_1975)
  df["Carbon price [$2020/tCO2e] (SAI demand feedback)"] = (
    solve_rf(rf, sulfur_cost=sulfur_cost_sai,
    fuel_cost=fuel_cost_constant).x[0]/US_2020_1975)
  df["Carbon price [$2020/tCO2e] (sulfur demand feedback)"] = (
    solve_rf(rf, sulfur_cost=sulfur_cost_demand,
    fuel_cost=fuel_cost_constant).x[0]/US_2020_1975)
  df["Carbon price [$2020/tCO2e] (variable fuel cost)"] = (
    solve_rf(rf, sulfur_cost=sulfur_cost_demand,
    fuel_cost=fuel_cost_variable).x[0]/US_2020_1975)

print(f"Results for model: {model}")
df.style.format(thousands=",", precision=2)

Results for model: CESM


,Radiative forcing [W/m2],Emissions [Tg SO2/yr],CO2 equivalent [GWP],Cumulative depletion [Tg SO2] (no demand feedback),Fossil supply [Mt S/yr],Sulfur demand [Mt S/yr],Mining supply [Mt S/yr],Cumulative depletion [Tg SO2] (demand feedback),Marginal sulfur cost (constant) [2020 USD/kg S],Marginal sulfur cost (no demand feedback) [2020 USD/kg S],Marginal sulfur cost (demand feedback) [2020 USD/kg S],Fuel cost (constant) [2020 USD/kg delivered SO2],Carbon price [$2020/tCO2e] (GWP fixed to 1 Tg),Carbon price [$2020/tCO2e] (variable GWP),Carbon price [$2020/tCO2e] (SAI demand feedback),Carbon price [$2020/tCO2e] (sulfur demand feedback)
Year,,,,,,,,,,,,,,,,
2100,-2.53,20.21,"-1,853.65","13,404.25",95.98,493.33,407.46,"45,524.36",4.26,4.90,56.48,0.45,3.90,7.45,8.22,58.07


In [109]:
df

,Radiative forcing [W/m2],Emissions [Tg SO2/yr],CO2 equivalent [GWP],Cumulative depletion [Tg SO2] (no demand feedback),Fossil supply [Mt S/yr],Sulfur demand [Mt S/yr],Mining supply [Mt S/yr],Cumulative depletion [Tg SO2] (demand feedback),Marginal sulfur cost (constant) [2020 USD/kg S],Marginal sulfur cost (no demand feedback) [2020 USD/kg S],Marginal sulfur cost (demand feedback) [2020 USD/kg S],Fuel cost (constant) [2020 USD/kg delivered SO2],Carbon price [$2020/tCO2e] (GWP fixed to 1 Tg),Carbon price [$2020/tCO2e] (variable GWP),Carbon price [$2020/tCO2e] (SAI demand feedback),Carbon price [$2020/tCO2e] (sulfur demand feedback),Fuel cost (increasing) [2020 USD/kg delivered SO2]
Year,,,,,,,,,,,,,,,,,
2100,-2.528335,20.211822,-1853.647345,13404.253322,95.980876,493.333587,407.458622,45524.360238,4.258577,4.896996,56.484879,0.448051,3.895011,7.453672,8.215492,58.071546,0.448723


In [108]:
df["Fuel cost (increasing) [2020 USD/kg delivered SO2]"] = (
  0.77155*0.15/US_2020_1975*(1+fuel_change)**(df.index-df.index[0]+1))
for row in df.iterrows():
  rf = row[1]["Radiative forcing [W/m2]"]
  sulfur_cost_const = (row[1][
    "Marginal sulfur cost (constant) [2020 USD/kg S]"]*US_2020_1975)
  sulfur_cost_sai = (row[1][
    "Marginal sulfur cost (no demand feedback) [2020 USD/kg S]"]*US_2020_1975)
  sulfur_cost_demand = (row[1][
    "Marginal sulfur cost (demand feedback) [2020 USD/kg S]"]*US_2020_1975)
  fuel_cost_constant=(row[1][
    "Fuel cost (constant) [2020 USD/kg delivered SO2]"]*US_2020_1975)
  fuel_cost_variable=(row[1][
    "Fuel cost (increasing) [2020 USD/kg delivered SO2]"]*US_2020_1975)
fuel_cost_increasing=(row[1][
    "Fuel cost (increasing) [2020 USD/kg delivered SO2]"]*US_2020_1975)
solve_rf(rf, sulfur_cost=sulfur_cost_demand, fuel_cost=fuel_cost_increasing).x[0]/US_2020_1975

np.float64(58.0715462691889)

In [ ]:
# SSP5-4.5 by mitigation and 2.5 additional by SAI (~1.9)

model="CESM"
initial_depletion=12500
sulfur_elasticity=0.5
fuel_change=0.0015
fossil_change=0.0036
fossil_fraction=0.9
demand_base=80
demand_change=0.023
fossil_base=demand_base*fossil_fraction
mining_base=demand_base*(1-fossil_fraction)
years=np.arange(2020, 2110, 10)
forcings=[-0.077795901, -0.202776003, -0.377084028, -0.625278416,
  -0.945167262, -1.320845384, -1.737155925, -2.155592258, -2.5283355]

US_2020_1975=0.2583022317

df = pd.DataFrame(columns=["Year",
  "Radiative forcing [W/m2]", "Emissions [Tg SO2/yr]", "CO2 equivalent [GWP]",
  "Cumulative depletion [Tg SO2] (no demand feedback)"])
year_0=2020
df["Year"] = years
years_delta = df["Year"] - df["Year"].shift(1)
years_delta.loc[0] = df.loc[0, "Year"] - year_0
df = df.set_index("Year")
df["Radiative forcing [W/m2]"] = forcings
df["Emissions [Tg SO2/yr]"] = [optimize.root_scalar(calc_RF, args=(x),
  method="newton", x0=1).root for x in df["Radiative forcing [W/m2]"]]
df["Fossil supply [Mt S/yr]"] = (
  fossil_base * (1 + fossil_change)**(df.index-df.index[0]))
df["Sulfur demand [Mt S/yr]"] = (
  demand_base * (1 + demand_change)**(df.index-df.index[0]))
df["Mining supply [Mt S/yr]"] = (df["Emissions [Tg SO2/yr]"]/2
  + df["Sulfur demand [Mt S/yr]"] - df["Fossil supply [Mt S/yr]"])
df["Cumulative depletion [Tg SO2] (no demand feedback)"] = (
  years_delta.values*df["Emissions [Tg SO2/yr]"].shift(1)/2).cumsum()

df.loc[years[0], "Cumulative depletion [Tg SO2] (no demand feedback)"] = 0
df["Cumulative depletion [Tg SO2] (no demand feedback)"] += (
  df["Emissions [Tg SO2/yr]"]/2+initial_depletion+
  mining_base*(df.index-df.index[0]+1))
df["Cumulative depletion [Tg SO2] (demand feedback)"] = (
  df["Emissions [Tg SO2/yr]"]+initial_depletion+
  df["Mining supply [Mt S/yr]"]*(df.index-df.index[0]+1))
df["CO2 equivalent [GWP]"] = calc_GWP(df["Emissions [Tg SO2/yr]"], model)
df["Marginal sulfur cost (constant) [2020 USD/kg S]"] = (
  0.22*5/US_2020_1975)
df["Marginal sulfur cost (no demand feedback) [2020 USD/kg S]"] = (
  df["Marginal sulfur cost (constant) [2020 USD/kg S]"]*
  (df["Cumulative depletion [Tg SO2] (no demand feedback)"]/initial_depletion)
  **(1/sulfur_elasticity))
df["Marginal sulfur cost (demand feedback) [2020 USD/kg S]"] = (
  df["Marginal sulfur cost (constant) [2020 USD/kg S]"]*
  (df["Cumulative depletion [Tg SO2] (demand feedback)"]/initial_depletion)
  **(1/sulfur_elasticity))

df["Fuel cost (constant) [2020 USD/kg delivered SO2]"] = (
  0.77155*0.15/US_2020_1975)
df["Fuel cost (increasing) [2020 USD/kg delivered SO2]"] = (
  0.77155*0.15/US_2020_1975*(1+fuel_change)**(df.index-df.index[0]+1))

df = df.iloc[[-1]]

for row in df.iterrows():
  rf = row[1]["Radiative forcing [W/m2]"]
  sulfur_cost_const = (row[1][
    "Marginal sulfur cost (constant) [2020 USD/kg S]"]*US_2020_1975)
  sulfur_cost_sai = (row[1][
    "Marginal sulfur cost (no demand feedback) [2020 USD/kg S]"]*US_2020_1975)
  sulfur_cost_demand = (row[1][
    "Marginal sulfur cost (demand feedback) [2020 USD/kg S]"]*US_2020_1975)
  fuel_cost_constant=(row[1][
    "Fuel cost (constant) [2020 USD/kg delivered SO2]"]*US_2020_1975)
  fuel_cost_variable=(row[1][
    "Fuel cost (increasing) [2020 USD/kg delivered SO2]"]*US_2020_1975)
  
  df["Carbon price [$2020/tCO2e] (GWP fixed to 1 Tg)"] = (
    solve_rf(rf, fixed_GWP=1, sulfur_cost=sulfur_cost_const,
    fuel_cost=fuel_cost_constant).x[0]/US_2020_1975)
  df["Carbon price [$2020/tCO2e] (variable GWP)"] = (
    solve_rf(rf, sulfur_cost=sulfur_cost_const,
    fuel_cost=fuel_cost_constant).x[0]/US_2020_1975)
  df["Carbon price [$2020/tCO2e] (SAI demand feedback)"] = (
    solve_rf(rf, sulfur_cost=sulfur_cost_sai,
    fuel_cost=fuel_cost_constant).x[0]/US_2020_1975)
  df["Carbon price [$2020/tCO2e] (sulfur demand feedback)"] = (
    solve_rf(rf, sulfur_cost=sulfur_cost_demand,
    fuel_cost=fuel_cost_constant).x[0]/US_2020_1975)
  df["Carbon price [$2020/tCO2e] (variable fuel cost)"] = (
    solve_rf(rf, sulfur_cost=sulfur_cost_demand,
    fuel_cost=fuel_cost_variable).x[0]/US_2020_1975)

print(f"Results for model: {model}")
df.style.format(thousands=",", precision=2)

## Scenarios

In [112]:
# SSP5-4.5 by SAI

# Model parameters
model="CESM"
initial_depletion=12500
sulfur_elasticity=0.5
fuel_change=-0.0082
fossil_change=0.0085
fossil_fraction=0.9
demand_base=80
demand_change=0.018
fossil_base=demand_base*fossil_fraction
mining_base=demand_base*(1-fossil_fraction)
years=np.arange(2020, 2110, 10)
forcings=[-0.077795901, -0.202776003, -0.377084028, -0.625278416,
  -0.945167262, -1.320845384, -1.737155925, -2.155592258, -2.5283355]

# Create model DataFrame
df = setup_model(model=model, initial_depletion=initial_depletion,
  sulfur_elasticity=sulfur_elasticity, fuel_change=fuel_change,
  fossil_change=fossil_change, fossil_fraction=fossil_fraction,
  demand_base=demand_base, demand_change=demand_change,
  years=years, forcings=forcings)
# Run all cases
#df = calc_scenario(df, cases=[True, True, True, True, True])

# Results
#df.to_csv(f"SSP545-SAI_{model}_all.csv")
df.style.format(thousands=",", precision=2)

,Radiative forcing [W/m2],Emissions [Tg SO2/yr],CO2 equivalent [GWP],Cumulative depletion [Tg SO2] (no demand feedback),Fossil supply [Mt S/yr],Sulfur demand [Mt S/yr],Mining supply [Mt S/yr],Cumulative depletion [Tg SO2] (demand feedback),Marginal sulfur cost (constant) [2020 USD/kg S],Marginal sulfur cost (no demand feedback) [2020 USD/kg S],Marginal sulfur cost (demand feedback) [2020 USD/kg S],Fuel cost [2020 USD/kg delivered SO2],Fuel cost (changing) [2020 USD/kg delivered SO2]
Year,,,,,,,,,,,,,
2020,-0.08,0.29,"-4,179.51","12,508.15",72.00,80.00,8.15,"12,508.44",4.26,4.26,4.26,0.45,0.44
2030,-0.20,0.89,"-3,571.53","12,589.91",78.36,95.62,17.71,"12,695.67",4.26,4.32,4.39,0.45,0.41
2040,-0.38,1.84,"-3,169.07","12,674.81",85.28,114.30,29.94,"13,130.60",4.26,4.38,4.70,0.45,0.38
2050,-0.63,3.39,"-2,834.05","12,764.80",92.81,136.62,45.51,"13,914.06",4.26,4.44,5.28,0.45,0.35
2060,-0.95,5.64,"-2,554.26","12,862.89",101.01,163.31,65.12,"15,175.44",4.26,4.51,6.28,0.45,0.32
2070,-1.32,8.61,"-2,322.43","12,972.60",109.93,195.20,89.57,"17,076.74",4.26,4.59,7.95,0.45,0.29
2080,-1.74,12.26,"-2,128.22","13,097.47",119.64,233.32,119.81,"19,820.66",4.26,4.68,10.71,0.45,0.27
2090,-2.16,16.30,"-1,971.73","13,240.79",130.21,278.89,156.83,"23,651.34",4.26,4.78,15.25,0.45,0.25
2100,-2.53,20.21,"-1,853.65","13,404.25",141.71,333.36,201.75,"28,862.25",4.26,4.90,22.70,0.45,0.23


In [113]:
# SSP5-4.5 by mitigation and 2.5 additional by SAI (~1.9)

# Model parameters
model="CESM"
initial_depletion=12500
sulfur_elasticity=0.5
fuel_change=0.0015
fossil_change=0.0036
fossil_fraction=0.9
demand_base=80
demand_change=0.023
fossil_base=demand_base*fossil_fraction
mining_base=demand_base*(1-fossil_fraction)
years=np.arange(2020, 2110, 10)
forcings=[-0.077795901, -0.202776003, -0.377084028, -0.625278416,
  -0.945167262, -1.320845384, -1.737155925, -2.155592258, -2.5283355]

# Create model DataFrame
df = setup_model(model=model, initial_depletion=initial_depletion,
  sulfur_elasticity=sulfur_elasticity, fuel_change=fuel_change,
  fossil_change=fossil_change, fossil_fraction=fossil_fraction,
  demand_base=demand_base, demand_change=demand_change,
  years=years, forcings=forcings)
# Run all cases
#df = calc_scenario(df, cases=[True, True, True, True, True])

# Results
#df.to_csv(f"SSP519-SAI_{model}_all.csv")
df.style.format(thousands=",", precision=2)

,Radiative forcing [W/m2],Emissions [Tg SO2/yr],CO2 equivalent [GWP],Cumulative depletion [Tg SO2] (no demand feedback),Fossil supply [Mt S/yr],Sulfur demand [Mt S/yr],Mining supply [Mt S/yr],Cumulative depletion [Tg SO2] (demand feedback),Marginal sulfur cost (constant) [2020 USD/kg S],Marginal sulfur cost (no demand feedback) [2020 USD/kg S],Marginal sulfur cost (demand feedback) [2020 USD/kg S],Fuel cost [2020 USD/kg delivered SO2],Fuel cost (changing) [2020 USD/kg delivered SO2]
Year,,,,,,,,,,,,,
2020,-0.08,0.29,"-4,179.51","12,508.15",72.00,80.00,8.15,"12,508.44",4.26,4.26,4.26,0.45,0.45
2030,-0.20,0.89,"-3,571.53","12,589.91",74.63,100.43,26.23,"12,789.47",4.26,4.32,4.46,0.45,0.46
2040,-0.38,1.84,"-3,169.07","12,674.81",77.37,126.07,49.62,"13,543.94",4.26,4.38,5.00,0.45,0.46
2050,-0.63,3.39,"-2,834.05","12,764.80",80.20,158.26,79.76,"14,975.81",4.26,4.44,6.11,0.45,0.47
2060,-0.95,5.64,"-2,554.26","12,862.89",83.13,198.66,118.35,"17,358.18",4.26,4.51,8.21,0.45,0.48
2070,-1.32,8.61,"-2,322.43","12,972.60",86.17,249.39,167.52,"21,052.05",4.26,4.59,12.08,0.45,0.48
2080,-1.74,12.26,"-2,128.22","13,097.47",89.32,313.06,229.87,"26,534.07",4.26,4.68,19.19,0.45,0.49
2090,-2.16,16.30,"-1,971.73","13,240.79",92.59,392.99,308.55,"34,423.39",4.26,4.78,32.30,0.45,0.50
2100,-2.53,20.21,"-1,853.65","13,404.25",95.98,493.33,407.46,"45,524.36",4.26,4.90,56.48,0.45,0.51


In [114]:
# SSP5-4.5 by SAI, all models

# Run for all models
df   = []
df_2 = []
for model in ["CESM", "IPSL", "MPI", "UKESM"]:
  # Model parameters
  initial_depletion=12500
  sulfur_elasticity=0.5
  fuel_change=0.0015
  fossil_change=0.0036
  fossil_fraction=0.9
  demand_base=80
  demand_change=0.023
  fossil_base=demand_base*fossil_fraction
  mining_base=demand_base*(1-fossil_fraction)
  years=np.arange(2020, 2110, 10)
  forcings=[-0.077795901, -0.202776003, -0.377084028, -0.625278416,
    -0.945167262, -1.320845384, -1.737155925, -2.155592258, -2.5283355]

  # Create model DataFrame
  df.append( setup_model(model=model, initial_depletion=initial_depletion,
    sulfur_elasticity=sulfur_elasticity, fuel_change=fuel_change,
    fossil_change=fossil_change, fossil_fraction=fossil_fraction,
    demand_base=demand_base, demand_change=demand_change,
    years=years, forcings=forcings) )
  # Run all cases
  #df[-1] = calc_scenario(df, cases=[False, True, False, False, False])

  # Save only carbon prices and emissions
  df_2.append(df[-1][["Emissions [Tg SO2/yr]",
    #"Carbon price [$2020/tCO2e] (variable GWP)"
    ]])
  df_2[-1].columns += f" - {model}"

  # Results
  #df[-1].to_csv(f"SSP545-SAI_{model}_variable_GWP.csv")

# Join model results
df_all=pd.concat(df_2, axis=1)
df_all.style.format(thousands=",", precision=2)

,Emissions [Tg SO2/yr] - CESM,Emissions [Tg SO2/yr] - IPSL,Emissions [Tg SO2/yr] - MPI,Emissions [Tg SO2/yr] - UKESM
Year,,,,
2020,0.29,0.29,0.29,0.29
2030,0.89,0.89,0.89,0.89
2040,1.84,1.84,1.84,1.84
2050,3.39,3.39,3.39,3.39
2060,5.64,5.64,5.64,5.64
2070,8.61,8.61,8.61,8.61
2080,12.26,12.26,12.26,12.26
2090,16.30,16.30,16.30,16.30
2100,20.21,20.21,20.21,20.21
